In [4]:
import os, time, json
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

ENV_FILE = find_dotenv(usecwd=True)          # sucht .env ab dem Notebook-Ordner (labs/.env)
print("Konfiguration aus:", ENV_FILE or "keine .env gefunden, Defaults (OpenAI, gpt-4.1)")
load_dotenv(ENV_FILE, override=True)   # override: CrewAI lädt beim Import schon eine .env aus einem Elternordner
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "https://api.openai.com/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4.1")

from pydantic import BaseModel
from crewai import Agent, Task, Crew, Process, LLM, TaskOutput

DATA = Path("data")
diff_text = (DATA / "diff_rabatt.patch").read_text(encoding="utf-8")
print(f"Diff geladen: {len(diff_text.splitlines())} Zeilen")
print(diff_text[:600])

llm = LLM(model=f"openai/{LLM_MODEL}", base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)
print("Modell:", llm.model, "| Endpunkt:", LLM_BASE_URL)
print("LLM antwortet:", llm.call("Antworte nur mit OK.")[:40])

Konfiguration aus: /Users/grigo/gits/seminars/allSkills/ki-agenten-crewai/src_live/.env
Diff geladen: 43 Zeilen
diff --git a/shop/rabatt.py b/shop/rabatt.py
index 3f1c2a9..8be47d1 100644
--- a/shop/rabatt.py
+++ b/shop/rabatt.py
@@ -1,12 +1,37 @@
 """Preisberechnung für den Webshop."""
 
 from decimal import Decimal
+
+# Rabattstaffel: ab Bestellwert (inklusive) gilt der Prozentsatz.
+# 0 bis 99,99 EUR: 0 %, 100 bis 499,99 EUR: 5 %, ab 500 EUR: 10 %
+STAFFEL = [
+    (Decimal("500"), Decimal("0.10")),
+    (Decimal("100"), Decimal("0.05")),
+]
+
+
+def rabattsatz(bestellwert: Decimal) -> Decimal:
+    """Liefert den Rabattsatz für einen Bestellwert."""
+    if bestellwert < 0:
+        raise ValueError(
Modell: gpt-4.1 | Endpunkt: https://api.openai.com/v1
LLM antwortet: OK.


In [ ]:
def make_llm(): 
    return LLM(model=f"openai/{LLM_MODEL}", base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)


In [8]:
# wir definieren zwei Agenten: Code Analyst und Technical Writer 
# role, goal, backstory => jeder Agent bekommt das 

def make_agents():
    analyst = Agent(
        role = "Code Analyst",
        goal = "Lese diff und liste jede funktionale Änderung und Risiko (bugs, fehlende Tests, Grenzwerte) auf. ",
        backstory = "Senior developer, welcher Code Reviews schreibt.",
        llm = make_llm(),
        max_iter = 3,
        verbose = True, 
    )

    writer = Agent(
        role = "Technical Writer",
        goal = "Die technische Analyse zusammenfassen für Entwickler.",
        backstory = "Kurz fassen, keine Marketing-Sprache, keine Spekulationen.",
        llm = make_llm(),
        max_iter = 3,
        verbose = True,
        )

    return analyst, writer

analyst, writer = make_agents()

In [14]:
# zwei Tasks 

class Zusammenfassung(BaseModel):
    title: str
    changes: list[str]
    risks: list[str]
    needs_tests: bool 

def make_tasks(analyst, writer, **task2_kwargs): 
    task1 = Task(
        description=(
            "Analyse this unified diff:\n\n{diff}\n\n"
            "List every functional change and every risk. Check boundary conditions "
            "against the comments in the code and check whether tests were added."
        ),
        expected_output=(
            "Two bullet lists titled 'Changes' and 'Risks'. "
            "Each bullet is one sentence of at most 20 words and names the affected function."
        ),
        agent=analyst,
    )
    task2 = Task(
        description=(
            "Write a changelog entry from the analysis you received as context. "
            "Write title, changes and risks in German. Each item in changes and risks is "
            "one sentence of at most 120 characters. "
            "Set needs_tests to true if the analysis reports missing tests."
        ),
        expected_output=(
            "A Zusammenfassung: title (one line), changes (list of short sentences), "
            "risks (list of short sentences), needs_tests (bool)."
        ),
        agent=writer,
        context=[task1],
        output_pydantic=Zusammenfassung,
        **task2_kwargs,
    )

    return task1, task2

task1, task2 = make_tasks(analyst, writer)

crew = Crew(agents=[analyst, writer], tasks=[task1,task2], process=Process.sequential, verbose=True)

result = await crew.kickoff_async(inputs={"diff": diff_text})
summary = result.pydantic

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 434ec298-a1f7-4de4-aea8-bab9969b4565                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyse this unified diff:                                                                               │
│                                                                                                                 │
│  diff --git a/shop/rabatt.py b/shop/rabatt.py                                                                   │
│  index 3f1c2a9..8be47d1 100644                                                                                  │
│  --- a/shop/rabatt.py                                                                                           │
│  +++ b/shop/rabatt.py                                                                                           │
│  @@ -1,12 +1,37 @@                                                                                              │
│   """Preisberechnung für den Webshop."""                                                                        │
│                                                                                                                 │
│   from decimal import Decimal                                                                                   │
│  +                                                                                                              │
│  +# Rabattstaffel: ab Bestellwert (inklusive) gilt der Prozentsatz.                                             │
│  +# 0 bis 99,99 EUR: 0 %, 100 bis 499,99 EUR: 5 %, ab 500 EUR: 10 %                                             │
│  +STAFFEL = [                                                                                                   │
│  +    (Decimal("500"), Decimal("0.10")),                                                                        │
│  +    (Decimal("100"), Decimal("0.05")),                                                                        │
│  +]                                                                                                             │
│  +                                                                                                              │
│  +                                                                                                              │
│  +def rabattsatz(bestellwert: Decimal) -> Decimal:                                                              │
│  +    """Liefert den Rabattsatz für einen Bestellwert."""                                                       │
│  +    if bestellwert < 0:                                                                                       │
│  +        raise ValueError("Bestellwert darf nicht negativ sein")                                               │
│  +    for grenze, satz in STAFFEL:                                                                              │
│  +        if bestellwert > grenze:                                                                              │
│  +            return satz                                                                                       │
│  +    return Decimal("0")                                                                                       │
│  +                                                                                                              │
│  +                                                                                                              │
│  +def rabattierter_preis(bestellwert: Decimal) -> Decimal:                                                      │
│  +    """Bestellwert abzüglich Staffelrabatt, auf Cent gerundet."""                                             │
│  +    preis = bestellwert * (Decimal("1") - rabattsatz(

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Code Analyst                                                                                            │
│                                                                                                                 │
│  Task: Analyse this unified diff:                                                                               │
│                                                                                                                 │
│  diff --git a/shop/rabatt.py b/shop/rabatt.py                                                                   │
│  index 3f1c2a9..8be47d1 100644                                                                                  │
│  --- a/shop/rabatt.py                                                                                           │
│  +++ b/shop/rabatt.py                                                                                           │
│  @@ -1,12 +1,37 @@                                                                                              │
│   """Preisberechnung für den Webshop."""                                                                        │
│                                                                                                                 │
│   from decimal import Decimal                                                                                   │
│  +                                                                                                              │
│  +# Rabattstaffel: ab Bestellwert (inklusive) gilt der Prozentsatz.                                             │
│  +# 0 bis 99,99 EUR: 0 %, 100 bis 499,99 EUR: 5 %, ab 500 EUR: 10 %                                             │
│  +STAFFEL = [                                                                                                   │
│  +    (Decimal("500"), Decimal("0.10")),                                                                        │
│  +    (Decimal("100"), Decimal("0.05")),                                                                        │
│  +]                                                                                                             │
│  +                                                                                                              │
│  +                                                                                                              │
│  +def rabattsatz(bestellwert: Decimal) -> Decimal:                                                              │
│  +    """Liefert den Rabattsatz für einen Bestellwert."""                                                       │
│  +    if bestellwert < 0:                                                                                       │
│  +        raise ValueError("Bestellwert darf nicht negativ sein")                                               │
│  +    for grenze, satz in STAFFEL:                                                                              │
│  +        if bestellwert > grenze:                                                                              │
│  +            return satz                                                                                       │
│  +    return Decimal("0")                                                                                       │
│  +                                                                                                              │
│  +                                                                                                              │
│  +def rabattierter_preis(bestellwert: Decimal) -> Decimal:                                                      │
│  +    """Bestellwert abzüglich Staffelrabatt, auf Cent 

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Code Analyst                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Changes                                                                                                        │
│                                                                                                                 │
│  - Added global variable STAFFEL in rabatt.py to define discount tiers based on order value.                    │
│  - Added function rabattsatz(bestellwert) to determine discount rate from STAFFEL for a given order value.      │
│  - Added function rabattierter_preis(bestellwert) to calculate discounted price and round to two decimal        │
│  places.                                                                                                        │
│  - Changed endpreis(netto) to apply rabattierter_preis before brutto, so discount is applied before VAT.        │
│  - Updated endpreis(netto) docstring to reflect new calculation order.                                          │
│                                                                                                                 │
│  Risks                                                                                                          │
│                                                                                                                 │
│  - rabattsatz: Discount thresholds in STAFFEL are checked with '>', which excludes exact boundary values        │
│  (e.g., 100, 500).                                                                                              │
│  - rabattsatz: Negative bestellwert raises ValueError, but no test coverage for this edge case is shown.        │
│  - rabattierter_preis: Rounding is to two decimals, but no test ensures correct rounding behavior at 0.005      │
│  steps.                                                                                                         │
│  - endpreis: Changing calculation order may break compatibility with previous pricing logic; no migration or    │
│  test coverage shown.                                                                                           │
│  - No tests were added for new functions or for boundary values (99.99, 100, 499.99, 500) as described in the   │
│  comments.                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Task: Write a changelog entry from the analysis you received as context. Write title, changes and risks in     │
│  German. Each item in changes and risks is one sentence of at most 120 characters. Set needs_tests to true if   │
│  the analysis reports missing tests.                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  title='Rabattberechnung und Reihenfolge der Preisermittlung angepasst' changes=['Globale Variable STAFFEL in   │
│  rabatt.py für Rabattstufen hinzugefügt.', 'Funktion rabattsatz(bestellwert) zur Ermittlung des Rabattsatzes    │
│  ergänzt.', 'Funktion rabattierter_preis(bestellwert) zur Berechnung des rabattierten Preises ergänzt.',        │
│  'endpreis(netto) wendet nun Rabatt vor Mehrwertsteuer an.', 'Docstring von endpreis(netto) an neue             │
│  Berechnungsreihenfolge angepasst.'] risks=["STAFFEL prüft mit '>', Grenzwerte wie 100 oder 500 werden          │
│  ausgeschlossen.", 'rabattsatz: Negativer Bestellwert wirft ValueError, aber kein Test für diesen Fall.',       │
│  'rabattierter_preis: Rundung auf zwei Stellen, aber kein Test für Rundungsgenauigkeit.', 'endpreis: Neue       │
│  Reihenfolge kann alte Preislogik brechen, keine Migration oder Tests.', 'Keine Tests für neue Funktionen oder  │
│  Grenzwerte wie 99.99, 100, 499.99, 500.'] needs_tests=True                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a changelog entry from the analysis you received as context. Write title, changes and risks in     │
│  German. Each item in changes and risks is one sentence of at most 120 characters. Set needs_tests to true if   │
│  the analysis reports missing tests.                                                                            │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 434ec298-a1f7-4de4-aea8-bab9969b4565                                                                       │
│  Final Output: {"title":"Rabattberechnung und Reihenfolge der Preisermittlung angepasst","changes":["Globale    │
│  Variable STAFFEL in rabatt.py für Rabattstufen hinzugefügt.","Funktion rabattsatz(bestellwert) zur Ermittlung  │
│  des Rabattsatzes ergänzt.","Funktion rabattierter_preis(bestellwert) zur Berechnung des rabattierten Preises   │
│  ergänzt.","endpreis(netto) wendet nun Rabatt vor Mehrwertsteuer an.","Docstring von endpreis(netto) an neue    │
│  Berechnungsreihenfolge angepasst."],"risks":["STAFFEL prüft mit '>', Grenzwerte wie 100 oder 500 werden        │
│  ausgeschlossen.","rabattsatz: Negativer Bestellwert wirft ValueError, aber kein Test für diesen                │
│  Fall.","rabattierter_preis: Rundung auf zwei Stellen, aber kein Test für Rundungsgenauigkeit.","endpreis:      │
│  Neue Reihenfolge kann alte Preislogik brechen, keine Migration oder Tests.","Keine Tests für neue Funktionen   │
│  oder Grenzwerte wie 99.99, 100, 499.99, 500."],"needs_tests":true}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
summary.changes, summary.risks

(['Globale Variable STAFFEL in rabatt.py für Rabattstufen hinzugefügt.',
  'Funktion rabattsatz(bestellwert) zur Ermittlung des Rabattsatzes ergänzt.',
  'Funktion rabattierter_preis(bestellwert) zur Berechnung des rabattierten Preises ergänzt.',
  'endpreis(netto) wendet nun Rabatt vor Mehrwertsteuer an.',
  'Docstring von endpreis(netto) an neue Berechnungsreihenfolge angepasst.'],
 ["STAFFEL prüft mit '>', Grenzwerte wie 100 oder 500 werden ausgeschlossen.",
  'rabattsatz: Negativer Bestellwert wirft ValueError, aber kein Test für diesen Fall.',
  'rabattierter_preis: Rundung auf zwei Stellen, aber kein Test für Rundungsgenauigkeit.',
  'endpreis: Neue Reihenfolge kann alte Preislogik brechen, keine Migration oder Tests.',
  'Keine Tests für neue Funktionen oder Grenzwerte wie 99.99, 100, 499.99, 500.'])